# Feature Pipeline Step-by-Step Evaluation

This notebook walks through one full pass of the FoehnCast feature pipeline for a single configured spot and an isolated review dataset.
The notebook is only the inspection surface; the components under review are the same configuration, ingest, engineering, validation, storage, and Feast paths used by the project.

It follows the same operational path used by the project: select a spot, review the storage contract and Feast boundary, fetch forecast data, engineer curated features, validate them, store them through the active backend, and prepare the resulting data for Feast offline use.

The walkthrough uses a separate `notebook_eval` review dataset so the exercise stays isolated from the default `train` dataset.
This notebook is a one-off design-validation document that can later be distilled into README and GitHub Pages content.

## Table of Contents

1. Review Scope, Pipeline Configuration, and Feast Boundary
2. Storage and Terraform Review
3. Ingest Raw Forecast Rows
4. Engineer Curated Features
5. Validate Curated Features
6. Store Curated Rows Through the Active Backend
7. Build the Feast Offline Frame
8. Export a Feast-Ready Parquet File
9. Documentation Handoff

## Pipeline Review Flow

```text
[Configured spot + isolated review dataset]
                |
                v
[Review config: spot metadata, API, storage, Feast repo, export path]
                |
                v
[Review storage contract: local store, Feast overlay, Terraform cloud path]
                |
                v
[Fetch raw forecast rows for the selected coordinates]
                |
                v
[Engineer curated feature frame]
                |
                v
[Validate schema, completeness, and ranges]
                |
                v
[Write curated rows via local / S3 / BigQuery backend]
                |
                +------------------------------+
                |                              |
                v                              v
[Build Feast offline frame]        [Inspect stored dataset state]
                |
                v
[Export Feast-ready parquet artifact]
```

The review starts by selecting one configured spot and one isolated review dataset name.
It then checks the real configuration values that shape the rest of the walkthrough: spot metadata, API settings, storage backend, Feast repo/config surfaces, and the planned Feast export destination.

Before ingest begins, the notebook reviews how the local filesystem store, the optional Feast layer, and the Terraform-managed cloud path fit together.
That keeps the storage and feature-serving discussion explicit instead of burying it inside later steps.

Once the configuration, storage, and Feast contract are fixed, the notebook fetches raw forecast rows for the selected coordinates.
Those rows are transformed into the curated feature frame, validated against the configured schema and ranges, and then written through the active storage backend for a round-trip check.

The last part of the walkthrough keeps Feast downstream of the curated store: it builds the offline frame and entity rows from stored features, exports a Feast-ready parquet file, and closes by identifying which stable findings belong in README or GitHub Pages rather than in the notebook itself.

In [ ]:
from pathlib import Path

import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from foehncast import config as config_module
from foehncast.config import get_api_config, get_spots, get_storage_config
from foehncast.feature_pipeline.engineer import engineer_features
from foehncast.feature_pipeline.feast import (
    build_entity_rows,
    build_offline_store_frame,
    export_offline_store,
)
from foehncast.feature_pipeline.ingest import fetch_forecast
from foehncast.feature_pipeline.store import (
    list_datasets,
    read_features,
    write_features,
)
from foehncast.feature_pipeline.validate import run_validation
from foehncast.paths import feast_offline_path, project_root as foehncast_project_root

pd.set_option("display.max_columns", 50)

In [ ]:
import base64
from IPython.display import Markdown, SVG
import requests


def render_mermaid(diagram: str, title: str | None = None) -> None:
    source = diagram.strip()
    encoded = base64.urlsafe_b64encode(source.encode("utf-8")).decode("ascii")
    endpoints = [
        ("GET", f"https://mermaid.ink/svg/{encoded}"),
        ("POST", "https://kroki.io/mermaid/svg"),
    ]
    last_error = None

    for method, url in endpoints:
        try:
            if method == "GET":
                response = requests.get(url, timeout=20)
            else:
                response = requests.post(url, data=source.encode("utf-8"), timeout=20)
            response.raise_for_status()
            if title:
                display(Markdown(f"### {title}"))
            display(SVG(data=response.text))
            return
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        "Could not render Mermaid from the notebook kernel. "
        "Check network access to mermaid.ink or kroki.io."
    ) from last_error


print("Mermaid renderer ready. Run a component-diagram cell to render it.")

## 1. Review Scope, Pipeline Configuration, and Feast Boundary

This step fixes the review scope without touching the default training dataset.
It validates the real configuration contract for the selected spot, API settings, storage backend, Feast hand-off, and isolated review dataset.

### Step 1 Component Diagram

Run the next cell to render the diagram from the notebook kernel.

In [ ]:
render_mermaid(
    """
    flowchart LR
        A[config.yaml] --> B[config.py loaders]
        B --> C[spot definitions]
        B --> D[Open-Meteo API config]
        B --> E[storage backend config]
        F[feature_store.yaml] --> G[Feast local contract]
        H[feature_store.gcp.yaml.example] --> I[Feast cloud contract]
        C --> J[selected spot]
        D --> K[reviewed pipeline contract]
        E --> K
        J --> K
        L[isolated review dataset] --> K
        G --> K
        I --> K
    """,
    title="Step 1 Component Diagram",
)

In [ ]:
import yaml

for _obsolete_name in ("local_file", "cloud_file", "stale_name"):
    globals().pop(_obsolete_name, None)
globals().pop("_obsolete_name", None)

spots = {spot["id"]: spot for spot in get_spots()}
spot_overview = pd.DataFrame(get_spots())[
    [
        "id",
        "name",
        "canton",
        "lat",
        "lon",
        "shore_orientation_deg",
        "ideal_wind_dir_min",
        "ideal_wind_dir_max",
    ]
]
spot_id = "silvaplana"
dataset = "notebook_eval"

if spot_id not in spots:
    raise KeyError(f"Unknown spot_id {spot_id!r}. Choose one of: {sorted(spots)}")

spot = spots[spot_id]
api_cfg = get_api_config()["open_meteo"]
storage_cfg = get_storage_config()
project_root = foehncast_project_root()
feast_repo_path = project_root / "feature_repo"
feast_state_path = project_root / ".state" / "feast"
feast_local_config_path = feast_repo_path / "feature_store.yaml"
feast_cloud_config_example_path = feast_repo_path / "feature_store.gcp.yaml.example"
feast_export_path = feast_offline_path(dataset, storage_cfg["local_path"])

feast_local_cfg = yaml.safe_load(feast_local_config_path.read_text())
feast_cloud_cfg = yaml.safe_load(feast_cloud_config_example_path.read_text())
feast_local_registry_path = (feast_repo_path / feast_local_cfg["registry"]).resolve()
feast_local_online_store_path = (
    feast_repo_path / feast_local_cfg["online_store"]["path"]
).resolve()
feast_local_registry_rel = feast_local_registry_path.relative_to(
    project_root
).as_posix()
feast_local_online_store_rel = feast_local_online_store_path.relative_to(
    project_root
).as_posix()

feast_contract_overview = pd.DataFrame(
    [
        {
            "surface": "provider",
            "local": feast_local_cfg["provider"],
            "cloud_example": feast_cloud_cfg["provider"],
        },
        {
            "surface": "offline_store",
            "local": f"{feast_local_cfg['offline_store']['type']} -> {feast_export_path.relative_to(project_root).as_posix()}",
            "cloud_example": feast_cloud_cfg["offline_store"]["type"],
        },
        {
            "surface": "online_store",
            "local": f"{feast_local_cfg['online_store']['type']} -> {feast_local_online_store_rel}",
            "cloud_example": feast_cloud_cfg["online_store"]["type"],
        },
        {
            "surface": "registry",
            "local": feast_local_registry_rel,
            "cloud_example": feast_cloud_cfg["registry"],
        },
    ]
)

display(spot_overview.set_index("id"))
display(feast_contract_overview.set_index("surface"))
print(f"Selected spot: {spot['name']} ({spot_id})")
print(f"Review dataset: {dataset}")
print(f"Shore orientation: {spot['shore_orientation_deg']} deg")
print(
    "Ideal wind window: "
    f"{spot['ideal_wind_dir_min']} -> {spot['ideal_wind_dir_max']} deg"
)
print(f"Storage backend: {storage_cfg['backend']}")
print(f"Forecast URL: {api_cfg['forecast_url']}")
print(f"Forecast horizon (days): {api_cfg['forecast_days']}")
print(f"Feast repo path: {feast_repo_path}")
print(f"Feast local config: {feast_local_config_path}")
print(f"Feast cloud example: {feast_cloud_config_example_path}")
print(f"Feast export path: {feast_export_path}")
print(f"Feast local state root: {feast_state_path}")
print(
    "Feast hand-off: local exported parquet -> Feast repo with runtime state under .state/feast; "
    "cloud curated BigQuery -> BigQuery/Datastore repo"
)
print(
    "Feast remains downstream of curated features and does not replace "
    "ingest, validation, or storage."
)

In [ ]:
step1_plot_df = spot_overview.sort_values("id").reset_index(drop=True)
selected_y = int(step1_plot_df.index[step1_plot_df["id"] == spot_id][0])
selected_row = step1_plot_df.loc[selected_y]

fig, ax = plt.subplots(figsize=(9, 4.5))
y_positions = list(range(len(step1_plot_df)))

ax.hlines(
    y=y_positions,
    xmin=step1_plot_df["ideal_wind_dir_min"],
    xmax=step1_plot_df["ideal_wind_dir_max"],
    color="#9fb3c8",
    linewidth=8,
    label="Ideal wind window",
)
ax.scatter(
    step1_plot_df["shore_orientation_deg"],
    y_positions,
    color="#1f4e79",
    s=90,
    label="Shore orientation",
)
ax.scatter(
    [selected_row["shore_orientation_deg"]],
    [selected_y],
    color="#c44900",
    s=130,
    label=f"Selected spot: {selected_row['name']}",
    zorder=3,
)

ax.set_xlim(0, 360)
ax.set_xlabel("Degrees")
ax.set_yticks(y_positions, step1_plot_df["id"])
ax.set_title("Configured spots: ideal wind window vs. shore orientation")
ax.grid(axis="x", alpha=0.3)
ax.legend(loc="lower right")
plt.show()

### Step 1 Review Summary

This step checks the current configuration and Feast boundary before any data is fetched or transformed.
The notebook is only the review surface. The real things under review are the spot config, API config, storage config, Feast hand-off, and the isolated review dataset.

#### Current Configuration Review

| Field group | Current representation | What we saw | Decision |
| --- | --- | --- | --- |
| Spot identifiers and labels | Strings such as `id`, `name`, `canton`, `difficulty`, `water_type` | Readable and easy to inspect | Keep strings for now. Add tighter validation in code where a field has a small fixed set of valid values. |
| Coordinates | `lat` and `lon` as decimal-degree floats | Fits configuration and API use | Keep floats in config and validate normal latitude and longitude bounds. |
| Altitude and forecast horizon | Integers such as `altitude_m` and `forecast_days` | Clear and easy to validate | Keep integers and keep explicit range checks. |
| URLs and paths | Strings such as API URLs and storage paths | Works today, but the meaning is only implicit | Keep YAML readable, then coerce to URL and Path types after loading. |
| Angular values | `shore_orientation_deg`, `ideal_wind_dir_min`, and `ideal_wind_dir_max` in degrees | Human-readable, but circular values need care in code | Keep degrees in YAML and normalize angles once after loading. |
| Direction ranges | Minimum and maximum degrees | Fine for simple windows, awkward across north such as `330 -> 30` | Keep the current config shape for now. Treat a dedicated direction-window helper as follow-up code work, not a notebook change in this pass. |

#### Angle and Direction Review

Wind direction is circular.
That means `359` degrees and `1` degree are close, even though plain subtraction says otherwise.

Decision in this review:
- keep degrees in configuration because they are easy to read;
- normalize angles into `[0, 360)` in code after loading;
- keep `shore_alignment` as the main circular feature because it already uses cosine similarity;
- leave richer direction-window handling as follow-up code work rather than changing the notebook story today.

#### Configuration Loading Review

The current loader returns nested dictionaries from YAML.
That works for the current project, but it spreads type cleanup across the codebase.

Decision in this review:
- keep YAML as the authoring format;
- keep the current loader in place for now;
- do not change the loading approach in this notebook review.

#### Feast Boundary Review

Step 1 should show Feast early because it is already part of the project boundary.
The current repo already shows two Feast modes: a lightweight local path with file offline storage plus SQLite, and a cloud-oriented path with BigQuery, Datastore, and GCS support.

Decision in this review:
- keep Feast downstream of curated features;
- keep the local Feast path lightweight for development;
- keep the cloud Feast path tied to curated BigQuery data, not to raw ingest outputs;
- keep Airflow, MLflow, and Feast as separate tools with different jobs.

#### Model Path and Preprocessing Review

The current training path is tree-based.
For this reason, the main preprocessing concern is not generic scaling.
It is keeping units, circular wind handling, and engineered feature definitions clear.

Decision in this review:
- prioritize explicit unit handling at ingest and labeling boundaries;
- keep circular feature handling visible;
- avoid broad preprocessing changes here unless the training path changes.

#### BigQuery and Location-Feature Review

The current BigQuery writer stores curated feature rows together with `spot_id`, `dataset_name`, and `forecast_time`.
That is a workable starting point, but location metadata still lives mainly in configuration.

Decision in this review:
- keep curated cloud storage in native BigQuery tables;
- keep raw landing separate if a landing layer is added later;
- add richer location-aware fields only when the warehouse use case needs them;
- keep Feast reading curated data instead of becoming the storage boundary.

This step shows that the current configuration, Feast boundary, and warehouse shape are clear enough to move on to the data-facing checks.

## 2. Storage, Terraform, and Feast Surface Review

This step checks where curated feature data lives, what Feast reads, and which cloud surfaces Terraform provides.
The main point is simple: raw landing, curated storage, and Feast do different jobs and should stay separate.
Local and cloud are also separate deployment targets. The local path should stay self-sufficient, and the cloud path should stay self-sufficient. One should not depend on the other for its core storage flow.

#### Storage Roles

| Surface | Local baseline | Cloud direction | What we saw today |
| --- | --- | --- | --- |
| Raw landing layer | Local files if retained at all | GCS object storage | Cheap, append-friendly, and suited to immutable raw payload capture. |
| Main curated feature store | `LocalFeatureStoreBackend -> data/<dataset>/<spot>.parquet` | `BigQueryFeatureStoreBackend -> <project>.<dataset>.<table>` | Matches the current adapter split in `feature_pipeline.store`. |
| Feast offline source | `export_offline_store -> data/feast/<dataset>.parquet` | BigQuery table or view over curated rows | Keeps Feast downstream from curated storage. |
| Feast registry | `.state/feast/registry.db` via `feature_store.yaml` | GCS registry path via `feature_store.gcp.yaml.example` | Keeps registry metadata separate from the curated dataset. |
| Feast online store | `SQLite -> .state/feast/online_store.db` | Datastore | Stays optional and downstream. |
| S3-compatible object storage | Optional experiment path via `S3FeatureStoreBackend` | Not the primary target | Useful for compatibility tests, not as the baseline path. |

#### Local Baseline and Optional S3 Path

Local S3-compatible storage is available, but it is not the baseline.
The current repo is easier to inspect with file-based local data, exported Feast parquet, and separate SQLite state for the optional local Feast layer.

Decision in this review:
- keep file-based local storage as the default;
- keep S3-compatible storage as an optional compatibility path;
- keep BigQuery as the main cloud curated store;
- keep Feast runtime state outside the workload data root.

#### Landing Layer

If the project keeps a raw landing layer, it should stay separate from curated features.
Raw payloads are cheapest to keep in files or object storage, while curated rows belong in parquet locally and native BigQuery tables in the cloud.

Decision in this review:
- keep raw landing separate from curated features;
- keep local raw landing feeding only the local path;
- keep GCS raw landing feeding only the cloud path;
- keep Feast on curated data only;
- do not treat BigQuery as the main raw landing surface.

#### Feast Boundary

Feast remains downstream of curated storage.
It can read local exported parquet or cloud BigQuery sources, but it should not own ingestion, raw retention, or schema drift handling.
Its local runtime state should stay under `.state/feast/`, not inside the workload dataset tree.

#### Current Storage Decisions

| Layer | Local decision | Cloud decision |
| --- | --- | --- |
| Raw landing | Files | GCS |
| Curated features | Parquet | BigQuery native tables |
| Feast offline | Exported parquet | BigQuery table or view |
| Feast registry and staging | Local state files | GCS |
| Feast online | SQLite under `.state/feast/` | Datastore |

The next cells render this split and inspect the concrete storage and Terraform surfaces from the live repository.

#### Pandera Decision

Pandera is optional in the current repo.
If the storage boundary later needs stronger reusable dataframe checks, the best fit is the curated write boundary.
We are not introducing it in this pass.

### Step 2 Component Diagram

Run the next cell to render the diagram from the notebook kernel, then inspect the concrete storage and Terraform surfaces in the following cell.

In [ ]:
render_mermaid(
    """
    flowchart LR
        ROOT[Same repo and pipeline contract]

        subgraph Local
            ROOT --> A[Local raw landing files]
            A --> B[Local curated feature rows]
            B --> D[LocalFeatureStoreBackend]
            D --> E[data/<dataset>/<spot>.parquet]
            E --> F[export_offline_store]
            F --> G[data/feast/<dataset>.parquet]
            G --> H[feature_store.yaml]
        end

        subgraph Cloud
            ROOT --> C[GCS raw landing]
            C --> I[Cloud curated feature rows]
            I --> J[BigQueryFeatureStoreBackend]
            J --> K[BigQuery curated table]
            K --> N[feature_store.gcp.yaml.example]
            L[Terraform] --> C
            L --> K
            L --> M[GCS registry and staging]
            N --> M
        end
    """,
    title="Step 2 Component Diagram",
)

In [ ]:
cloud_project = (
    storage_cfg.get("bigquery_project_id")
    or os.getenv("GCP_PROJECT_ID")
    or os.getenv("GOOGLE_CLOUD_PROJECT")
    or "<gcp-project>"
)
cloud_dataset = storage_cfg.get("bigquery_dataset", "foehncast")
cloud_table = storage_cfg.get("bigquery_table", "forecast_features")
cloud_bigquery_target = f"{cloud_project}.{cloud_dataset}.{cloud_table}"
cloud_registry_target = feast_cloud_cfg["registry"]
cloud_online_store = feast_cloud_cfg["online_store"]["type"]
cloud_raw_target = os.getenv("GCP_BUCKET_NAME") or "GCS object storage"
terraform_root = project_root / "terraform"
terraform_files = sorted(path.name for path in terraform_root.glob("*.tf"))

storage_surface_overview = pd.DataFrame(
    [
        {
            "surface": "runtime backend",
            "local_or_current": storage_cfg["backend"],
            "cloud_or_target": "local / s3 / bigquery",
            "implementation": "config.yaml -> storage.backend",
        },
        {
            "surface": "curated feature store",
            "local_or_current": (
                f"LocalFeatureStoreBackend -> {storage_cfg['local_path']}/{dataset}/{spot_id}.parquet"
            ),
            "cloud_or_target": (
                f"BigQueryFeatureStoreBackend -> {cloud_bigquery_target}"
            ),
            "implementation": "feature_pipeline.store adapters",
        },
        {
            "surface": "Feast offline source",
            "local_or_current": (
                f"export_offline_store -> {feast_export_path.relative_to(project_root).as_posix()}"
            ),
            "cloud_or_target": f"BigQuery offline store -> {cloud_bigquery_target}",
            "implementation": "feature_pipeline.feast + feature_repo config",
        },
        {
            "surface": "Feast registry",
            "local_or_current": feast_local_registry_rel,
            "cloud_or_target": cloud_registry_target,
            "implementation": "feature_store.yaml variants",
        },
        {
            "surface": "Feast online store",
            "local_or_current": f"{feast_local_cfg['online_store']['type']} -> {feast_local_online_store_rel}",
            "cloud_or_target": cloud_online_store,
            "implementation": "feature_store.yaml variants",
        },
        {
            "surface": "raw landing direction",
            "local_or_current": "local files only if retained",
            "cloud_or_target": cloud_raw_target,
            "implementation": "storage review target, not Feast",
        },
    ]
)
terraform_surface_overview = pd.DataFrame({"terraform_file": terraform_files})

display(storage_surface_overview.set_index("surface"))
display(terraform_surface_overview)
print(f"Active runtime backend: {storage_cfg['backend']}")
print(f"Cloud curated target: {cloud_bigquery_target}")
print(
    "Boundary reminder: curated rows live in the main store first; Feast reads "
    "a local exported parquet artifact or a curated BigQuery table/view downstream."
)
print(
    "Local Feast state reminder: registry and online-store SQLite files live under "
    f"{feast_state_path.relative_to(project_root).as_posix()}, not under the workload data root."
)

## 3. Ingest Raw Forecast Rows

This stage exercises the live ingest path for one configured spot and shows the raw forecast shape,
time range, and column coverage before any engineering, curated storage, or Feast shaping happens.

### Step 3 Component Diagram

Run the next cell to render the diagram from the notebook kernel.

In [ ]:
render_mermaid(
    """
    flowchart LR
        A[Selected spot lat and lon] --> B[fetch_forecast]
        B --> C[Open-Meteo forecast API]
        C --> D[raw_df]
        D --> E[Column and timestamp checks]
    """,
    title="Step 3 Component Diagram",
)

In [ ]:
raw_df = fetch_forecast(spot["lat"], spot["lon"])

expected_columns = [column.strip() for column in api_cfg["hourly_params"].split(",")]
missing_columns = sorted(set(expected_columns) - set(raw_df.columns))
unexpected_columns = sorted(set(raw_df.columns) - set(expected_columns))
index_timezone = str(raw_df.index.tz) if raw_df.index.tz is not None else "naive"

ingest_checks = pd.DataFrame(
    [
        {
            "rows": raw_df.shape[0],
            "columns": raw_df.shape[1],
            "expected_columns": len(expected_columns),
            "missing_expected": len(missing_columns),
            "unexpected_columns": len(unexpected_columns),
            "index_timezone": index_timezone,
            "monotonic_index": raw_df.index.is_monotonic_increasing,
            "duplicate_timestamps": int(raw_df.index.duplicated().sum()),
            "total_null_values": int(raw_df.isna().sum().sum()),
        }
    ]
)

display(ingest_checks)
print(f"Time range: {raw_df.index.min()} -> {raw_df.index.max()}")
print(f"Columns ({len(raw_df.columns)}): {sorted(raw_df.columns.tolist())}")
print("Missing expected columns:", missing_columns or "none")
print("Unexpected columns:", unexpected_columns or "none")
display(raw_df.head())

In [ ]:
plot_df = raw_df[["wind_speed_10m", "wind_gusts_10m", "wind_direction_10m"]].head(48)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, height_ratios=[2, 1])
plot_df[["wind_speed_10m", "wind_gusts_10m"]].plot(ax=axes[0], linewidth=2)
axes[0].set_ylabel("Wind (km/h)")
axes[0].set_title("Raw ingest preview: first 48 forecast hours")
axes[0].grid(alpha=0.3)

plot_df["wind_direction_10m"].plot(ax=axes[1], color="#c44900", linewidth=1.75)
axes[1].set_ylabel("Direction (deg)")
axes[1].set_ylim(0, 360)
axes[1].grid(alpha=0.3)
axes[1].set_xlabel("Forecast time")

plt.tight_layout()
plt.show()

### Step 3 Review Summary

This step checks the live ingest path before feature engineering starts.
It uses the same Open-Meteo helper as the pipeline, so the review is tied to the real source path.

What we saw:
- expected column coverage, timestamp order, duplicate timestamps, null counts, and index timezone are visible here;
- raw forecast rows stay separate from spot metadata at this stage;
- Feast is not part of ingest, which keeps this step focused on the source contract;
- timezone handling is explicit before storage and Feast use later.

Decision in this review:
- keep ingest focused on source-contract fidelity;
- keep raw weather rows separate from downstream enrichment;
- keep timestamp and timezone checks explicit at the ingest boundary;
- if a raw landing layer is added later, preserve the upstream payload without mixing in feature logic.

## 4. Engineer Curated Features

This step checks the same transformation layer the batch feature pipeline uses before validation, storage, and Feast projection.

### Step 4 Component Diagram

Run the next cell to render the diagram from the notebook kernel.

In [ ]:
render_mermaid(
    """
    flowchart LR
        A[raw_df] --> B[engineer_features]
        C[shore orientation] --> B
        B --> D[feature_df]
        D --> E[Engineered feature checks]
        D --> F[Validation boundary]
        F --> G[Storage adapter]
        G --> H[Feast projection later]
    """,
    title="Step 4 Component Diagram",
)

In [ ]:
feature_df = engineer_features(raw_df, spot["shore_orientation_deg"])
new_columns = sorted(set(feature_df.columns) - set(raw_df.columns))
preview_columns = new_columns + [
    "wind_speed_10m",
    "wind_gusts_10m",
    "wind_direction_10m",
]
engineered_columns = [
    "hour_of_day_sin",
    "hour_of_day_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    "wind_steadiness",
    "gust_factor",
    "shore_alignment",
]
preserved_raw_columns = sorted(set(raw_df.columns) - set(feature_df.columns))
index_timezone = (
    str(feature_df.index.tz) if feature_df.index.tz is not None else "naive"
)
feast_projection_ready = (
    isinstance(feature_df.index, pd.DatetimeIndex) or "time" in feature_df.columns
)

feature_checks = pd.DataFrame(
    [
        {
            "rows_match_raw": len(feature_df) == len(raw_df),
            "raw_columns_preserved": len(preserved_raw_columns) == 0,
            "preserves_datetime_index": isinstance(feature_df.index, pd.DatetimeIndex),
            "feast_projection_ready": feast_projection_ready,
            "engineered_column_count": len(new_columns),
            "all_expected_features_present": set(engineered_columns).issubset(
                feature_df.columns
            ),
            "nulls_in_engineered_features": int(
                feature_df[engineered_columns].isna().sum().sum()
            ),
            "shore_alignment_min": float(feature_df["shore_alignment"].min()),
            "shore_alignment_max": float(feature_df["shore_alignment"].max()),
            "gust_factor_min": float(feature_df["gust_factor"].min()),
            "gust_factor_max": float(feature_df["gust_factor"].max()),
        }
    ]
)

engineered_ranges = feature_df[engineered_columns].agg(["min", "max", "mean"]).T

display(feature_checks)
print(f"Engineered shape: {feature_df.shape}")
print(f"New feature columns ({len(new_columns)}): {new_columns}")
print(f"Missing raw columns after engineering: {preserved_raw_columns or 'none'}")
print(f"Feature index timezone: {index_timezone}")
print(
    "Feast hand-off readiness: "
    f"{'yes' if feast_projection_ready else 'no'}; engineering keeps the time basis but does not add serving metadata yet."
)
display(engineered_ranges)
display(feature_df[preview_columns].head())

In [ ]:
engineer_plot_df = feature_df[
    [
        "shore_alignment",
        "gust_factor",
        "wind_steadiness",
    ]
].head(48)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
engineer_plot_df["shore_alignment"].plot(ax=axes[0], color="#1f4e79", linewidth=1.8)
axes[0].set_ylabel("alignment")
axes[0].set_ylim(-1.05, 1.05)
axes[0].set_title("Engineered feature preview: first 48 forecast hours")
axes[0].grid(alpha=0.3)

engineer_plot_df["gust_factor"].plot(ax=axes[1], color="#c44900", linewidth=1.8)
axes[1].set_ylabel("gust factor")
axes[1].grid(alpha=0.3)

engineer_plot_df["wind_steadiness"].plot(ax=axes[2], color="#3b7a57", linewidth=1.8)
axes[2].set_ylabel("steadiness")
axes[2].set_xlabel("Forecast time")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Step 4 Review Summary

This step turns raw forecast rows into the curated feature frame used later in the pipeline.

What we saw:
- cyclical time features use sine and cosine, which fits hour-of-day and day-of-year better than plain integers;
- `shore_alignment` already uses circular math instead of plain angle subtraction;
- `gust_factor` and `wind_steadiness` depend on rolling and denominator behavior, so they need visible checks;
- row count, raw forecast columns, and the datetime index are preserved through engineering;
- raw wind values stay in source weather units, while downstream label thresholds stay in knots and are handled later at scoring time.

Decision in this review:
- keep engineering focused on feature creation rather than Feast or serving metadata;
- keep the current tree-based preprocessing path simple in this step;
- treat this curated frame as the base contract for validation, storage, and Feast preparation.

## 5. Validate Curated Features

Validation covers schema, completeness, and configured numeric ranges. A failing result here is the same signal that would stop the Airflow-managed feature refresh job.

### Step 5 Component Diagram

Run the next cell to render the diagram from the notebook kernel.

In [ ]:
render_mermaid(
    """
    flowchart LR
        A[feature_df] --> B[run_validation]
        C[validation config] --> B
        B --> D[ValidationResult]
        D --> E[Schema, completeness, and range review]
        D --> F[Storage gate]
        D --> G[Feast projection gate]
    """,
    title="Step 5 Component Diagram",
)

In [ ]:
validation_cfg = config_module.get_validation_config()
validation = run_validation(feature_df, spot_id)

required_columns = validation_cfg["required_columns"]
raw_columns = set(raw_df.columns)
engineered_validation_columns = [
    column for column in required_columns if column not in raw_columns
]
validated_columns = [
    column for column in required_columns if column in feature_df.columns
]
engineered_range_columns = [
    column
    for column in engineered_validation_columns
    if column in validation_cfg["ranges"] and column in feature_df.columns
]

summary = pd.DataFrame(
    [
        {
            "spot_id": validation.spot_id,
            "validated_columns": len(validated_columns),
            "engineered_columns_required": len(engineered_validation_columns),
            "engineered_range_checks": len(engineered_range_columns),
            "schema_valid": validation.schema_valid,
            "completeness_valid": validation.completeness_valid,
            "range_valid": validation.range_valid,
            "is_valid": validation.is_valid,
            "missing_columns": len(validation.missing_columns),
            "range_violations": len(validation.range_violations),
        }
    ]
)

null_summary = (
    pd.Series(validation.null_fractions, name="null_fraction")
    .sort_values(ascending=False)
    .rename_axis("column")
)

range_summary = pd.DataFrame(
    [
        {
            "column": column,
            "configured_min": bounds.get("min"),
            "observed_min": float(feature_df[column].min()),
            "observed_max": float(feature_df[column].max()),
            "configured_max": bounds.get("max"),
        }
        for column, bounds in validation_cfg["ranges"].items()
        if column in feature_df.columns
    ]
).set_index("column")

display(summary)
print(f"Required curated columns ({len(required_columns)}): {required_columns}")
print(f"Completeness threshold: {validation_cfg['completeness']['max_null_pct']}")
print(
    f"Engineered columns covered by range checks: {engineered_range_columns or 'none'}"
)
print("Missing columns:", validation.missing_columns or "none")
display(null_summary.head(10))
display(range_summary)

if validation.range_violations.empty:
    print("No range violations detected.")
else:
    display(validation.range_violations.head(20))

print(
    "Validation gate: ",
    "pass" if validation.is_valid else "fail",
    "-> downstream storage and Feast projection should only proceed on a valid curated frame.",
)

In [ ]:
engineered_range_plot = range_summary.loc[
    [
        column
        for column in engineered_validation_columns
        if column in range_summary.index
    ]
]

fig, axes = plt.subplots(2, 1, figsize=(11, 8), constrained_layout=True)

null_summary.head(10).sort_values().plot.barh(ax=axes[0], color="#3b7a57")
axes[0].axvline(
    validation_cfg["completeness"]["max_null_pct"],
    color="#c44900",
    linestyle="--",
    linewidth=1.5,
    label="max null pct",
)
axes[0].set_title("Top null fractions across curated columns")
axes[0].set_xlabel("Null fraction")
axes[0].set_ylabel("Column")
axes[0].legend()

for position, (column, row) in enumerate(engineered_range_plot.iterrows()):
    bound_min = (
        row["configured_min"]
        if pd.notna(row["configured_min"])
        else row["observed_min"]
    )
    bound_max = (
        row["configured_max"]
        if pd.notna(row["configured_max"])
        else row["observed_max"]
    )
    axes[1].hlines(position, bound_min, bound_max, color="#d9d9d9", linewidth=6)
    axes[1].plot(
        [row["observed_min"], row["observed_max"]],
        [position, position],
        color="#1f4e79",
        linewidth=2.2,
        marker="o",
    )

axes[1].set_yticks(range(len(engineered_range_plot)))
axes[1].set_yticklabels(engineered_range_plot.index)
axes[1].set_title("Engineered feature ranges vs validation bounds")
axes[1].set_xlabel("Feature value")
axes[1].grid(axis="x", alpha=0.25)

plt.show()

### Step 5 Review Summary

This step checks that validation matches the curated feature frame, not only the raw ingest fields.

What we saw:
- engineered columns are included in the validation contract;
- cyclical features and `shore_alignment` have clear bounded ranges;
- `wind_steadiness` and `gust_factor` are lower-bounded at zero rather than clipped too aggressively;
- completeness still matters because some engineered signals can go null when sustained wind is close to zero;
- storage and Feast projection should only run after this gate passes.

Decision in this review:
- keep this layer structural, not semantic;
- use it to stop missing columns, null-heavy outputs, and impossible values;
- do not use it here to judge rideability or spot quality.

## 6. Store Curated Rows Through the Active Backend

This stage persists the validated feature rows and reads them back through the same storage adapter. That keeps this review aligned with the repo's local, S3-compatible, or BigQuery-backed storage modes.

### Step 6 Component Diagram

Run the next cell to render the diagram from the notebook kernel.

In [ ]:
render_mermaid(
    """
    flowchart LR
        A[validated feature_df] --> B[write_features]
        B --> C[Storage backend adapter]
        C --> D[read_features]
        D --> E[stored_df]
        E --> F[Round-trip checks]
        E --> G[Feast-ready stored frame]
    """,
    title="Step 6 Component Diagram",
)

In [ ]:
write_features(feature_df, spot_id=spot_id, dataset=dataset)
stored_df = read_features(spot_id=spot_id, dataset=dataset)

visible_datasets = list_datasets()
missing_after_roundtrip = sorted(set(feature_df.columns) - set(stored_df.columns))
extra_after_roundtrip = sorted(set(stored_df.columns) - set(feature_df.columns))
shared_columns = [
    column for column in feature_df.columns if column in stored_df.columns
]
numeric_shared_columns = (
    feature_df[shared_columns].select_dtypes(include="number").columns.tolist()
)

roundtrip_deltas = pd.Series(
    {
        column: float((stored_df[column] - feature_df[column]).abs().max())
        for column in numeric_shared_columns
    },
    name="max_abs_delta",
).sort_values(ascending=False)

original_index_tz = str(getattr(feature_df.index, "tz", None))
stored_index_tz = str(getattr(stored_df.index, "tz", None))
feast_roundtrip_ready = (
    isinstance(stored_df.index, pd.DatetimeIndex) or "time" in stored_df.columns
)
max_numeric_abs_delta = (
    float(roundtrip_deltas.max()) if not roundtrip_deltas.empty else 0.0
)
roundtrip_contract_valid = (
    validation.is_valid
    and not missing_after_roundtrip
    and not extra_after_roundtrip
    and list(stored_df.columns) == list(feature_df.columns)
    and feature_df.index.equals(stored_df.index)
    and original_index_tz == stored_index_tz
    and max_numeric_abs_delta == 0.0
)

roundtrip_summary = pd.DataFrame(
    [
        {
            "backend": storage_cfg["backend"],
            "validation_gate_passed": validation.is_valid,
            "rows_match_original": len(stored_df) == len(feature_df),
            "columns_match_original": list(stored_df.columns)
            == list(feature_df.columns),
            "missing_columns": len(missing_after_roundtrip),
            "extra_columns": len(extra_after_roundtrip),
            "index_equal": feature_df.index.equals(stored_df.index),
            "index_type": type(stored_df.index).__name__,
            "index_tz_matches": original_index_tz == stored_index_tz,
            "feast_roundtrip_ready": feast_roundtrip_ready,
            "roundtrip_contract_valid": roundtrip_contract_valid,
            "max_numeric_abs_delta": max_numeric_abs_delta,
            "datasets_visible": len(visible_datasets),
        }
    ]
)

display(roundtrip_summary)
print(f"Datasets visible through the storage adapter: {visible_datasets}")
print("Missing columns after read:", missing_after_roundtrip or "none")
print("Extra columns after read:", extra_after_roundtrip or "none")
print(f"Original index timezone: {original_index_tz}")
print(f"Stored index timezone: {stored_index_tz}")
print(
    "Feast hand-off after round-trip: "
    f"{'ready' if feast_roundtrip_ready else 'not ready'}; the stored frame still carries the time basis needed downstream."
)

if storage_cfg["backend"] == "local":
    print(
        "Local path:", Path(storage_cfg["local_path"]) / dataset / f"{spot_id}.parquet"
    )
elif storage_cfg["backend"] == "bigquery":
    print(
        "BigQuery semantics: writes replace one logical spot_id + dataset_name slice before append, so reruns do not accumulate duplicates for the same curated slice."
    )

display(roundtrip_deltas.head(10).to_frame())
display(stored_df.head())

In [ ]:
roundtrip_plot_columns = [
    column
    for column in ["wind_speed_10m", "gust_factor"]
    if column in feature_df.columns and column in stored_df.columns
]

fig, axes = plt.subplots(
    len(roundtrip_plot_columns),
    1,
    figsize=(11, 3.6 * len(roundtrip_plot_columns)),
    constrained_layout=True,
    squeeze=False,
)

for axis, column in zip(axes.flatten(), roundtrip_plot_columns, strict=False):
    feature_df[column].head(24).plot(
        ax=axis, color="#1f4e79", linewidth=2.0, label="original"
    )
    stored_df[column].head(24).plot(
        ax=axis, color="#c44900", linewidth=1.6, linestyle="--", label="stored"
    )
    axis.set_title(f"Round-trip comparison for {column}")
    axis.set_xlabel("")
    axis.grid(alpha=0.25)
    axis.legend()

plt.show()

### Step 6 Review Summary

This step checks that storage behaves like persistence, not transformation.

What we saw:
- round-trip reads are expected to return the same curated schema, index behavior, and numeric values that validation approved;
- backend-specific metadata may exist during writes, but it should not leak after read-back;
- BigQuery reruns replace one logical `spot_id + dataset_name` slice instead of stacking duplicates;
- the persisted pipeline summary now lives at the pipeline surface, and the notebook reads it as a client;
- that summary records the source-unit contract: raw wind features stay in km/h, while downstream rideability thresholds stay in knots and are converted later at scoring time.

Decision in this review:
- keep validation before storage;
- treat storage as a round-trip contract only;
- keep the notebook as a consumer of the persisted pipeline summary, not the monitoring source.

## 7. Build the Feast Offline Frame

This step validates the actual Feast preparation helpers by reading stored curated rows and reshaping them into the offline frame and entity rows used for historical retrieval.

In [ ]:
import importlib

from foehncast.monitoring import pipeline_metrics as pipeline_metrics_module

pipeline_metrics = importlib.reload(pipeline_metrics_module)
feature_pipeline_stage_overview = pipeline_metrics.feature_pipeline_stage_overview
feature_pipeline_summary_path = pipeline_metrics.feature_pipeline_summary_path
read_feature_pipeline_run_summary = pipeline_metrics.read_feature_pipeline_run_summary

summary_path = feature_pipeline_summary_path(dataset=dataset)

try:
    pipeline_run_summary = read_feature_pipeline_run_summary(dataset=dataset)
except FileNotFoundError:
    print(
        "No persisted feature-pipeline summary found for "
        f"{dataset!r}. Run run_feature_pipeline(dataset=dataset) or "
        "run_feature_pipeline_job(dataset=dataset) first."
    )
else:
    print(f"Summary path: {summary_path}")
    print(f"Run status: {pipeline_run_summary['run_status']}")
    print(
        "Stored spots: "
        f"{pipeline_run_summary['stored_spot_count']} / "
        f"{pipeline_run_summary['expected_spot_count']}"
    )
    spot_overview = feature_pipeline_stage_overview(pipeline_run_summary).set_index(
        "spot_id"
    )
    display(spot_overview)

    unit_contract_columns = [
        "wind_speed_10m_unit",
        "wind_gusts_10m_unit",
        "source_unit_contract_confirmed",
    ]
    available_unit_columns = [
        column for column in unit_contract_columns if column in spot_overview.columns
    ]

    if available_unit_columns:
        display(spot_overview[available_unit_columns])
        print(
            "Source-unit contract confirmed across displayed spots:",
            bool(spot_overview["source_unit_contract_confirmed"].all()),
        )
    else:
        print(
            "This persisted summary predates the explicit source-unit fields. "
            "Rerun the feature pipeline for this dataset to refresh the artifact."
        )

### Step 7 Component Diagram

Run the next cell to render the diagram from the notebook kernel.

In [ ]:
render_mermaid(
    """
    flowchart LR
        A[Stored curated rows] --> B[build_offline_store_frame]
        A --> C[build_entity_rows]
        B --> D[offline_df]
        C --> E[entity_rows]
        D --> F[Feast historical retrieval inputs]
        E --> F
    """,
    title="Step 7 Component Diagram",
)

In [ ]:
offline_df = build_offline_store_frame(dataset=dataset)
entity_rows = build_entity_rows(dataset=dataset)

print(f"Offline frame shape: {offline_df.shape}")
print(f"Entity row shape: {entity_rows.shape}")
print(f"Offline columns ({len(offline_df.columns)}): {offline_df.columns.tolist()}")
display(offline_df.head())
display(entity_rows.head())

### Step 7 Review Summary

This step checks the Feast preparation helpers, not Feast as a separate feature-engineering layer.

What we saw:
- `build_offline_store_frame(dataset=...)` and `build_entity_rows(dataset=...)` read stored curated features and reshape them for retrieval;
- the offline frame and entity rows stay thin projections of the stored feature set;
- Feast does not reach back into raw ingest or recompute feature logic here.

Decision in this review:
- keep Feast downstream of curated storage;
- keep these helpers deterministic and narrow;
- keep the same conceptual contract for local and cloud Feast paths.

## 8. Export a Feast-Ready Parquet File

This mirrors the export step behind `scripts/prepare-feast-local.sh`, but keeps the intermediate frame
visible in this design-review artifact so you can inspect the real export contract before applying or materializing the Feast repo.

### Step 8 Component Diagram

Run the next cell to render the diagram from the notebook kernel.

In [ ]:
render_mermaid(
    """
    flowchart LR
        A[offline_df] --> B[export_offline_store]
        B --> C[Feast-ready parquet file]
        C --> D[feature_repo and materialization flow]
    """,
    title="Step 8 Component Diagram",
)

In [ ]:
exported_path = export_offline_store(dataset=dataset, output_path=feast_export_path)
exported_df = pd.read_parquet(exported_path)

print(f"Exported Feast parquet: {exported_path}")
print(f"Exported shape: {exported_df.shape}")
display(exported_df.head())

### Step 8 Review Summary

This step checks that the export helper is a materialization step, not another place where feature logic changes.

What we saw:
- `export_offline_store(...)` writes the same offline-frame contract that the local Feast flow expects;
- the parquet artifact is visible here for inspection, but the real component under review is the export helper itself;
- the local script is an operator wrapper around the same helper.

Decision in this review:
- keep export thin and predictable;
- do not let feature semantics drift at export time;
- keep scripts as wrappers around the same validated helper.

## 9. Documentation Handoff

This notebook is review evidence. Public docs should reuse the stable findings from it, not the per-run inspection details.

### What moves out of the notebook

| Target | Reuse from this review | Keep only here |
| --- | --- | --- |
| README | short pipeline overview, stage responsibilities, local-first storage baseline, Feast boundary, and main operator entrypoints | intermediate inspection tables, plots, and run-specific outputs |
| GitHub Pages | architecture diagrams, storage and cloud target design, curated feature contract, validation boundary, and Feast hand-off | per-run dataset snapshots, local execution noise, and ad hoc inspection cells |
| Notebook | full evidence trail, helper execution, intermediate frames, and review notes | not applicable |

### Stable findings to reuse

- Step 1 for configuration scope and Feast boundary.
- Step 2 for storage roles and the raw-versus-curated split.
- Steps 3 to 6 for ingest, engineering, validation, and storage round-trip contracts.
- Steps 7 and 8 for Feast preparation and export boundaries.

### Step 9 Component Diagram

Run the next cell to render the diagram from the notebook kernel.

The batch helper `run_feature_pipeline(dataset=...)` applies the same ingest -> engineer -> validate -> store loop across all configured spots.
The current training path remains tree-based, so this notebook records the current feature contract rather than proposing a new preprocessing stack.

### Operator Steps for Local Feast

If you want to load this isolated review dataset into the local Feast repo, run:

- `uv sync --group feast`
- `./scripts/prepare-feast-local.sh notebook_eval`
- `cd feature_repo && uv run --group feast feast materialize-incremental "$(date -u +"%Y-%m-%dT%H:%M:%S")"`

In [ ]:
render_mermaid(
    """
    flowchart LR
        A[Ingest] --> B[Engineer]
        B --> C[Validate]
        C --> D[Store]
        D --> E[Build Feast offline frame]
        E --> F[Export Feast parquet]
        F --> G[Materialize and serve]
    """,
    title="Step 9 Component Diagram",
)